# Online Retail — Câu 10: Tạo luật kết hợp bằng Apriori

**Võ Thiện Linh — MSSV: 6451071041**

Tìm các luật kết hợp từ tập phổ biến Apriori ở Câu 9, với `min_conf = 50%`. Hiển thị độ hỗ trợ, độ tin cậy và lift của từng luật.

In [ ]:
print('Võ Thiện Linh - MSSV: 6451071041')
from pathlib import Path
from collections import Counter
from itertools import combinations
import math
import pandas as pd
from IPython.display import display
candidates = [Path('Online Retail.xlsx'), Path('LAB1') / 'Phan02' / '6451071041_LAB1_Bai2_Trang37' / 'Online Retail.xlsx', Path('/content/Online Retail.xlsx')]
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    raise FileNotFoundError(f'Không tìm thấy Online Retail.xlsx. Thư mục đang chạy: {Path.cwd()}')
df = pd.read_excel(data_path, engine='openpyxl')
df['Description'] = df['Description'].str.strip()
df = df.dropna(subset=['InvoiceNo']).copy()
df['InvoiceNo'] = df['InvoiceNo'].astype('string').str.strip()
df = df.loc[~df['InvoiceNo'].str.contains('C', regex=False, na=False)]
uk = df.loc[df['Country'].eq('United Kingdom')]
grouped = uk.groupby(['InvoiceNo', 'Description'], as_index=False, dropna=True)['Quantity'].sum()
invoice_ids = pd.Index(grouped['InvoiceNo'].drop_duplicates(), name='InvoiceNo')
purchases = grouped.loc[grouped['Quantity'].gt(0) & grouped['Description'].ne('POSTAGE')]
items_by_invoice = purchases.groupby('InvoiceNo')['Description'].agg(frozenset).to_dict()
transactions = [items_by_invoice.get(invoice, frozenset()) for invoice in invoice_ids]
def apriori_counts(transactions, min_support):
    minimum_count = math.ceil(min_support * len(transactions))
    item_counts = Counter(item for transaction in transactions for item in transaction)
    previous = {frozenset([item]): count for item, count in item_counts.items() if count >= minimum_count}
    frequent_counts = dict(previous)
    frequent_items = set(item_counts)
    size = 2
    while previous:
        previous_list = sorted(previous, key=lambda itemset: tuple(sorted(itemset)))
        previous_set = set(previous)
        candidates = set()
        for i, left in enumerate(previous_list):
            for right in previous_list[i + 1:]:
                candidate = left | right
                if len(candidate) == size and all(frozenset(part) in previous_set for part in combinations(candidate, size - 1)):
                    candidates.add(candidate)
        if not candidates:
            break
        candidate_counts = Counter()
        for transaction in transactions:
            active_items = sorted(transaction & frequent_items)
            for combination in combinations(active_items, size):
                itemset = frozenset(combination)
                if itemset in candidates:
                    candidate_counts[itemset] += 1
        previous = {itemset: count for itemset, count in candidate_counts.items() if count >= minimum_count}
        frequent_counts.update(previous)
        size += 1
    return frequent_counts, minimum_count
def make_rules(itemset_counts, transaction_count, min_confidence):
    rules = []
    for itemset, itemset_count in itemset_counts.items():
        if len(itemset) < 2:
            continue
        for antecedent_size in range(1, len(itemset)):
            for left in combinations(sorted(itemset), antecedent_size):
                antecedent = frozenset(left)
                consequent = itemset - antecedent
                confidence = itemset_count / itemset_counts[antecedent]
                if confidence >= min_confidence:
                    support = itemset_count / transaction_count
                    consequent_support = itemset_counts[consequent] / transaction_count
                    rules.append({'antecedent': antecedent, 'consequent': consequent, 'support': support, 'confidence': confidence, 'lift': confidence / consequent_support})
    return rules
min_support = 0.03
min_confidence = 0.50
frequent_counts, minimum_count = apriori_counts(transactions, min_support)
rules = make_rules(frequent_counts, len(transactions), min_confidence)
rules_df = pd.DataFrame([{'antecedent': ', '.join(sorted(rule['antecedent'])), 'consequent': ', '.join(sorted(rule['consequent'])), 'support': rule['support'], 'confidence': rule['confidence'], 'lift': rule['lift']} for rule in rules])
if not rules_df.empty:
    rules_df = rules_df.sort_values(['confidence', 'support'], ascending=False).reset_index(drop=True)
print(f'Minimum support: {min_support:.0%}; minimum confidence: {min_confidence:.0%}')
print(f'Số tập phổ biến: {len(frequent_counts):,}; số luật đạt ngưỡng: {len(rules_df):,}')
display(rules_df)